# Topic agenda results

This notebook presents saved results from [15_topic_agenda_calculations.ipynb](15_topic_agenda_calculations.ipynb). The default `ATTENTION_MODEL='spline'` is primary; select `power_law` for sensitivity analysis. Choose one distance metric at the top; the default is cosine. `NORMALIZE_REFERENCE_GAINS=False` is selected here to show raw gains toward article versus relative-votes targets.

The notebook performs no expensive distribution, metric, or oracle calculations. If a required artifact is missing, rerun notebook 15.

**Conditional estimand:** topic distributions describe only passages/comments assigned a valid topic by the development article-trained model. Unassigned material is excluded and assigned mass is renormalized. Alignment results therefore concern the classifiable portion of eligible stories, not the entire article or discussion. Assigned-exposure coverage should be read alongside every policy comparison.


Select the topic run in notebook 14, or pin `TOPIC_RUN_ID` below. Calculations and figures stay under the selected run's `analysis/` directory. Set `LEGACY_TOPIC_ROOT` explicitly to use an older unregistered fit.

Calculation manifests are checked against the current topic-modeling, metric, policy, and pin/reply code. Set `ALLOW_OLDER_RUN_RESULTS=True` only when intentionally viewing a compatible historical calculation; stored provenance remains unchanged. Missing random-reference gains always require rerunning notebook 15.


In [ ]:
# The empirical spline is primary; both parametric models are fitted sensitivities.
ATTENTION_MODEL = 'spline'  # 'spline' or 'power_law'
ATTENTION_LABELS = {
    'spline': 'Smoothed empirical attention (primary)',
    'power_law': 'Fitted power-law attention (sensitivity)',
}
if ATTENTION_MODEL not in ATTENTION_LABELS:
    raise ValueError(f'ATTENTION_MODEL must be one of {list(ATTENTION_LABELS)}')
PREFIX = f'vote_{ATTENTION_MODEL}_'
ATTENTION_LABEL = ATTENTION_LABELS[ATTENTION_MODEL]
NORMALIZE_REFERENCE_GAINS = False  # Show raw gains toward article versus relative-votes targets.
DISTANCE_METRIC = 'cosine'  # 'jsd' or 'cosine'
DISTANCE_METRIC_CONFIG = {
    'jsd': {
        'metric_column': 'article_visible_js_distance',
        'metric_label': 'Discussion-Article Agenda Alignment',
        'x_label': 'Change in Jensen-Shannon distance',
        'metric_stem': 'article_visible_js_distance',
        'gain_column': 'js_raw_target_gain',
        'scatter_stem': 'js',
        'oracle_name': 'js_oracle',
    },
    'cosine': {
        'metric_column': 'cosine_progress',
        'metric_label': 'Discussion-Article Agenda Alignment',
        'x_label': 'Change in cosine similarity',
        'metric_stem': 'cosine_progress',
        'gain_column': 'cosine_raw_target_gain',
        'scatter_stem': 'cosine',
        'oracle_name': 'cosine_oracle',
    },
}
if DISTANCE_METRIC not in DISTANCE_METRIC_CONFIG:
    raise ValueError("DISTANCE_METRIC must be one of: 'jsd', 'cosine'")


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from commentgap_analysis.topic_policy import (
    build_policy_contrast_order,
)
from commentgap_analysis.topic_plotting import (
    plot_article_relative_votes_marginal_effects,
    plot_article_relative_votes_progress_scatter,
    plot_topic_policy_concentration_effects,
    plot_topic_policy_exposure_coverage,
    plot_topic_policy_metric_effects,
)
from commentgap_analysis.forum_scores import policy_specs
from commentgap_analysis.topic_artifacts import (
    code_revision, file_sha256, input_signature, required, topic_calculation_artifacts,
    topic_calculation_config, topic_calculation_source_manifests, validate_topic_run,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent
from commentgap_analysis.topic_runs import resolve_run
RUNS_ROOT = REPO_ROOT / 'model_output/selection_2025/paper2_topic_runs'
TOPIC_RUN_ID = 'mcs15_nn10_ms5_seed2026_1e1b8fce5deb3e58'  # Final no-outlier-reduction model.
LEGACY_TOPIC_ROOT = None  # Set to the old artifact directory to explicitly use a legacy fit.
ALLOW_OLDER_RUN_RESULTS = False  # Opt in when inspecting a compatible older run.
TOPIC_ROOT = (Path(LEGACY_TOPIC_ROOT) if LEGACY_TOPIC_ROOT is not None
              else resolve_run(RUNS_ROOT, TOPIC_RUN_ID))
print('Using topic model:', TOPIC_ROOT)
OUTPUT_ROOT = TOPIC_ROOT / 'analysis'


MEMBERSHIP_PATH = TOPIC_ROOT / 'document_topic_memberships.parquet'
COMMENTS_PATH = REPO_ROOT / 'model_output/selection_2025/paper2/analysis_comments.parquet'
INPUT_SIGNATURE = input_signature(
    repo_root=REPO_ROOT, membership_path=MEMBERSHIP_PATH,
    comments_path=COMMENTS_PATH, topic_root=TOPIC_ROOT,
)
SCORE_POLICIES = {
    'chronological': None, 'reverse_chronological': None, 'relative_votes': None, 'upvotes': None,
    'random': None,
    'regression_audience': 'regression_audience_score', 'regression_editor': 'regression_editor_score',
    'xgb_metadata_audience': 'xgb_metadata_audience_score', 'xgb_metadata_editor': 'xgb_metadata_editor_score',
    'xgb_metadata_text_audience': 'xgb_metadata_text_audience_score', 'xgb_metadata_text_editor': 'xgb_metadata_text_editor_score',
    'neural_metadata_audience': 'neural_metadata_audience_score', 'neural_metadata_editor': 'neural_metadata_editor_score',
    'neural_metadata_text_audience': 'neural_metadata_text_audience_score', 'neural_metadata_text_editor': 'neural_metadata_text_editor_score',
}
from commentgap_analysis.presentation_labels import ORDERING_DISPLAY_LABELS
ORDERING_LABELS = dict(ORDERING_DISPLAY_LABELS)
REPLY_LABELS = {'loose': 'Loose replies', 'trees': 'Thread trees', 'hidden': 'Hidden replies'}
contrast_order = build_policy_contrast_order(SCORE_POLICIES)
ANALYSIS_MODELS = ('spline', 'power_law')
DISTANCE_MEASURES = ('cosine', 'jensen_shannon')
CALCULATION_CONFIG = topic_calculation_config(
    score_policies=SCORE_POLICIES, policy_specs=policy_specs(),
    analysis_models=ANALYSIS_MODELS, distance_measures=DISTANCE_MEASURES,
)
CALCULATION_CODE = {
    'topic_artifacts_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_artifacts.py'),
    'topic_modeling_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_modeling.py'),
    'topic_metrics_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_metrics.py'),
    'topic_policy_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_policy.py'),
    'forum_scores_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/forum_scores.py'),
    'vote_attention_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/vote_attention.py'),
    'code_revision': code_revision(),
}
CALCULATION_PROVENANCE = {
    'run_id': TOPIC_RUN_ID, 'inputs': INPUT_SIGNATURE,
    'config': CALCULATION_CONFIG, 'code': CALCULATION_CODE,
}
SCHEMA_CHECKS = {
    f'{model}_metrics': ('story_id', 'policy_id') for model in ANALYSIS_MODELS
}
SCHEMA_CHECKS.update({
    f'{model}_metrics_draws': ('story_id', 'policy_id', 'draw') for model in ANALYSIS_MODELS
})
validate_topic_run(
    OUTPUT_ROOT, expected_provenance=CALCULATION_PROVENANCE,
    required_artifacts=topic_calculation_artifacts(OUTPUT_ROOT),
    optional_artifacts=topic_calculation_source_manifests(OUTPUT_ROOT),
    schema_checks=SCHEMA_CHECKS,
    allow_older_results=ALLOW_OLDER_RUN_RESULTS, allow_legacy_missing=ALLOW_OLDER_RUN_RESULTS,
)
metrics = pd.read_parquet(required(OUTPUT_ROOT / f'topic_policy_{PREFIX}metrics.parquet'))
concentration_story = pd.read_parquet(required(OUTPUT_ROOT / f'{PREFIX}topic_policy_concentration_story_metrics.parquet'))
coverage_summary = pd.read_csv(required(OUTPUT_ROOT / f'{PREFIX}topic_policy_exposure_coverage_summary.csv'))
reference_target_metrics = pd.read_parquet(required(OUTPUT_ROOT / f'{PREFIX}topic_policy_reference_target_metrics.parquet'))
random_gains = reference_target_metrics.loc[
    reference_target_metrics.policy_id.eq('random__loose__unpinned'),
    ['js_raw_target_gain', 'cosine_raw_target_gain'],
]
if random_gains.empty or random_gains.isna().any().any():
    raise RuntimeError('Reference gains are missing the random baseline. Rerun notebook 15.')
if random_gains.abs().max().max() > 1e-10:
    if ALLOW_OLDER_RUN_RESULTS:
        print('Allowing older random-baseline gains under explicit override')
    else:
        raise RuntimeError('Reference gains use an outdated random baseline. Rerun notebook 15.')
cosine_oracle = pd.read_parquet(required(OUTPUT_ROOT / f'topic_policy_{PREFIX}cosine_oracle_metrics.parquet'))
js_oracle = pd.read_parquet(required(OUTPUT_ROOT / f'topic_policy_{PREFIX}js_oracle_metrics.parquet'))
DISTANCE_CONFIG = DISTANCE_METRIC_CONFIG[DISTANCE_METRIC]
DISTANCE_ORACLE = cosine_oracle if DISTANCE_METRIC == 'cosine' else js_oracle

## Coverage

Coverage is the share of the policy’s total rank attention weight carried by comments with valid topic assignments. It is shown alongside the alignment metrics so a policy that appears more article-aligned can be checked for whether that result is partly driven by which comments received topics.

In [ ]:
for name in ['topic_agenda_baseline_summary.csv', 'topic_agenda_rarefaction_summary.csv']:
    path = OUTPUT_ROOT / name
    if path.exists():
        display(pd.read_csv(path).round(4))
display(coverage_summary.round(4))
coverage_plot = plot_topic_policy_exposure_coverage(
    coverage_summary, output_root=OUTPUT_ROOT, ordering_labels=ORDERING_LABELS,
    output_prefix=PREFIX, exposure_label=ATTENTION_LABEL,
)
display(coverage_plot.round(4))

In [ ]:
sensitivity_path = OUTPUT_ROOT / 'topic_policy_fitted_attention_sensitivity.csv'
if sensitivity_path.exists():
    display(pd.read_csv(sensitivity_path).round(4))

## Topic concentration and metric effects

These figures and tables are generated from the story-level artifacts. The two metric panels use the same policy contrasts.

In [ ]:
concentration_summary = pd.read_csv(required(OUTPUT_ROOT / f'{PREFIX}topic_policy_concentration_summary.csv'))
display(concentration_summary.round(4))
concentration_effects = plot_topic_policy_concentration_effects(
    concentration_story, contrast_order=contrast_order, output_root=OUTPUT_ROOT,
    ordering_labels=ORDERING_LABELS, reply_labels=REPLY_LABELS,
    output_stem=f'{PREFIX}topic_policy_concentration_effects',
)
metric_effects = plot_topic_policy_metric_effects(
    metrics, metric=DISTANCE_CONFIG['metric_column'], oracle_frame=DISTANCE_ORACLE,
    contrast_order=contrast_order, output_root=OUTPUT_ROOT,
    output_stem=f'{PREFIX}topic_policy_marginal_effects_{DISTANCE_CONFIG["metric_stem"]}',
    metric_label=DISTANCE_CONFIG['metric_label'],
    x_label=DISTANCE_CONFIG['x_label'],
    ordering_labels=ORDERING_LABELS, reply_labels=REPLY_LABELS,
)
display(metric_effects.round(4))

## Article and vote gains

Reference-target gains compare each policy with a random loose/unpinned presentation. The article target is its assigned-topic distribution; the relative-votes target is its mean assigned-topic distribution across tie draws.

With `NORMALIZE_REFERENCE_GAINS=True`, the marginal-effect figure compares the fractions of the two target gaps closed. This notebook defaults to `False` for raw gains. The scatter figures retain raw gains and their oracle markers use the same draw-averaged random baseline.


In [ ]:
random_oracle_base = metrics.loc[metrics['policy_id'].eq('random__loose__unpinned'), ['story_id', 'article_visible_js_distance', 'article_visible_cosine']].copy()
random_oracle_base['story_id'] = random_oracle_base['story_id'].astype(str)
oracle_gains = {
    'js_raw_target_gain': (
        random_oracle_base.set_index('story_id')['article_visible_js_distance']
        - js_oracle.assign(story_id=js_oracle['story_id'].astype(str)).set_index('story_id')['article_visible_js_distance']
    ).dropna().mean(),
    'cosine_raw_target_gain': (
        cosine_oracle.assign(story_id=cosine_oracle['story_id'].astype(str)).set_index('story_id')['article_visible_cosine']
        - random_oracle_base.set_index('story_id')['article_visible_cosine']
    ).dropna().mean(),
}
reference_effects = plot_article_relative_votes_marginal_effects(
    reference_target_metrics, contrast_order=contrast_order, output_root=OUTPUT_ROOT,
    ordering_labels=ORDERING_LABELS, reply_labels=REPLY_LABELS, output_prefix=PREFIX,
    normalize_gains=NORMALIZE_REFERENCE_GAINS,
    oracle_gains=oracle_gains,
    distance_metric=DISTANCE_METRIC,
)
scatter_summary = plot_article_relative_votes_progress_scatter(
    reference_target_metrics, metrics, DISTANCE_ORACLE, score_policies=SCORE_POLICIES,
    output_root=OUTPUT_ROOT, ordering_labels=ORDERING_LABELS, output_prefix=PREFIX,
    distance_metric=DISTANCE_METRIC,
)
display(reference_effects.round(4))
scatter_display_columns = [
    'policy_id', 'ordering', 'reply_mode', 'pinned', 'n_stories',
    f"{DISTANCE_CONFIG['scatter_stem']}_gain_article",
    f"{DISTANCE_CONFIG['scatter_stem']}_gain_relative_votes",
]
display(scatter_summary[scatter_display_columns].round(4))

## Saved figures

All output paths below share the selected attention-model prefix. The CSV summaries beside them are the presentation tables used above.

In [ ]:
figure_stems = [
    f'{PREFIX}topic_policy_exposure_coverage_relationship.png',
    f'{PREFIX}topic_policy_concentration_effects.png',
    f'{PREFIX}topic_policy_marginal_effects_{DISTANCE_CONFIG["metric_stem"]}.png',
    f'{PREFIX}topic_policy_article_vs_relative_votes_{"normalized" if NORMALIZE_REFERENCE_GAINS else "raw"}_gain_effects.png',
    f'{PREFIX}topic_policy_{DISTANCE_CONFIG["scatter_stem"]}_article_relative_votes_raw_gain_scatter.png',
]
for figure in figure_stems:
    path = OUTPUT_ROOT / figure
    if path.exists(): print(path)